# PharmaMate 2.2 — VS Code 개인 학습 Agent

**제약·바이오 기사 읽기 → Agent 질문 → 학습 기록 누적**을 분리해 사용하기 쉽게 구성했습니다.

### 핵심 변경

1. **기사 읽기:** 기사 입력, 브리핑, 원문, 핵심 용어 미리보기만 표시
2. **Agent 질문:** 학습할 용어 선택, 쉬운 설명, 추천 질문 3개, 자유 채팅, 대화 기반 학습 메모 작성·저장
3. **내 학습 기록:** 저장한 용어·기사·학습 메모·해당 용어와 관련된 Q&A를 조회하고 복습
4. **대화별 용어 연결:** LangGraph의 라우터가 질문이 현재 선택한 용어와 관련되는지 판단. 다른 용어 질문도 자유롭게 가능하며, 무관한 대화는 선택 용어의 학습 기록에 저장하지 않음
5. **로컬 영구 저장:** SQLite를 VS Code 프로젝트 폴더에 저장. Google Drive/Colab 불필요

### 실행

- Python 3.11 권장, VS Code의 **Python + Jupyter** 확장 설치
- 폴더 내 `.env`에 `OPENAI_API_KEY`(필수), `TAVILY_API_KEY`(선택) 입력
- 가상환경 선택 후 모든 셀을 **위에서 아래로** 실행, 마지막 Gradio URL 열기
- 기존 2.1 데이터베이스가 있으면 **그대로 재사용하며 새 테이블만 추가**합니다.

> 2.1에서 2.2로 변경 후에는 기존 Gradio 서버를 종료하고 마지막 셀을 다시 실행하세요. 화면이 유난히 작다면 Chrome에서 `Cmd+0` (Windows `Ctrl+0`)으로 브라우저 확대/축소를 100%로 초기화하세요.


## 1. VS Code 로컬 환경 준비

### 1-1. 폴더와 가상환경

노트북이 있는 프로젝트 폴더를 VS Code에서 열고 **터미널**을 실행합니다.

**Windows (PowerShell)**

```powershell
py -3.11 -m venv .venv
.\.venv\Scripts\python.exe -m pip install -U pip
.\.venv\Scripts\python.exe -m pip install -r requirements.txt
```

**macOS / Linux**

```bash
python3.11 -m venv .venv
./.venv/bin/python -m pip install -U pip
./.venv/bin/python -m pip install -r requirements.txt
```

- 노트북 **단독 파일**만 가지고 있다면 아래의 설치 셀을 실행해도 됩니다.
- 오른쪽 상단 **Select Kernel** → `.venv`의 Python을 선택합니다.
- `requirements.txt` 파일에는 이 프로젝트에서 사용하는 라이브러리가 정리되어 있습니다.

### 1-2. API 키 파일 만들기

프로젝트 폴더에서 `.env.example`을 복사하여 `.env`로 이름을 바꾼 뒤 API 키를 채웁니다.

```dotenv
OPENAI_API_KEY=여기에_OpenAI_API_키
TAVILY_API_KEY=여기에_Tavily_API_키_선택
PHARMAMATE_MODEL=gpt-4o-mini
```

- `.env`는 인터넷에 공유하거나 Git에 올리지 마세요 (`.gitignore` 포함).
- `TAVILY_API_KEY`가 없으면 최신 웹 검색만 비활성화되고 다른 기능은 동작합니다.
- VS Code에서 커널을 새로 선택하거나 환경 변수를 수정한 뒤에는 해당 설정 셀을 다시 실행합니다.

In [ ]:
# 가상환경에서 requirements.txt로 설치했다면 이 셀은 건너뛰어도 됩니다.
# 노트북만 받은 경우, 현재 선택한 Jupyter 커널에 필요한 패키지를 설치합니다.
# 설치 직후 import 오류가 나면 VS Code에서 노트북 커널을 재시작하세요.
%pip install -q "gradio>=5,<7" "langchain-openai>=0.3,<2" "langchain-core>=0.3,<2" "langgraph>=0.3,<2" "langchain-community>=0.3,<2" "tavily-python>=0.5,<1" "beautifulsoup4>=4.12" "requests>=2.31" "python-dotenv>=1.0" "pydantic>=2,<3" "ipykernel>=6"

## 2. `.env` 설정 및 모델 선택

프로젝트 최상위 폴더(이 노트북을 실행하는 **작업 폴더**)에 `.env`를 만들어 주세요.

- 필수: `OPENAI_API_KEY`
- 선택: `TAVILY_API_KEY` (최신 검색), `PHARMAMATE_MODEL` (기본값 `gpt-4o-mini`)

코드가 키를 화면에 출력하지 않으며, 없는 키를 실행 중에 입력받지 않습니다. `.env`를 작성하고 이 셀을 **다시 실행**하세요.

> VS Code Jupyter의 작업 폴더가 의도와 다르면 셀 출력에 표시되는 `PROJECT_DIR`을 확인하세요. 원하는 폴더를 탐색기에서 **폴더 열기**로 다시 열면 됩니다.

In [ ]:
import os
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

PROJECT_DIR = Path.cwd().resolve()
ENV_FILE = PROJECT_DIR / ".env"
load_dotenv(dotenv_path=ENV_FILE, override=False)

api_key = os.getenv("OPENAI_API_KEY", "").strip()
if not api_key or api_key.lower().startswith(("replace_", "your_", "여기에_")):
    raise RuntimeError(
        "OPENAI_API_KEY가 설정되지 않았습니다. "
        f"{ENV_FILE} 파일을 만들고 OPENAI_API_KEY=... 를 입력한 뒤 이 셀을 다시 실행하세요."
    )

MODEL_NAME = os.getenv("PHARMAMATE_MODEL", "gpt-4o-mini").strip() or "gpt-4o-mini"
print("프로젝트 폴더:", PROJECT_DIR)
print(".env 파일:", ENV_FILE, "(확인됨)" if ENV_FILE.is_file() else "(없음: 시스템 환경변수 사용 중)")
print("OpenAI 키: 설정됨 | Tavily 검색:", "사용 가능" if os.getenv("TAVILY_API_KEY") else "비활성화")
print("LLM 모델:", MODEL_NAME)

## 3. 로컬 SQLite 학습 데이터 저장

학습 데이터는 컴퓨터의 프로젝트 폴더 아래 `pharmamate_data/pharmamate.sqlite3`에 저장됩니다.

- **Google Drive 연결/승인 없음.** Colab 세션 종료에 따른 데이터 유실도 없습니다.
- VS Code를 껐다 켜도 저장된 기록을 계속 사용할 수 있습니다.
- 저장 위치를 바꾸려면 `.env`에 `PHARMAMATE_DATA_DIR=원하는_절대_경로`를 설정하세요.
- 다른 컴퓨터로 옮길 때는 **SQLite 파일도 함께 복사**해야 기존 학습 기록을 이어갈 수 있습니다.
- 보안을 위해 `.gitignore`에서 `.env`, `pharmamate_data/`를 제외합니다.

In [ ]:
# 기본값: 현재 VS Code 프로젝트 폴더 / pharmamate_data
_custom_data_dir = os.getenv("PHARMAMATE_DATA_DIR", "").strip()
DATA_DIR = Path(_custom_data_dir).expanduser().resolve() if _custom_data_dir else PROJECT_DIR / "pharmamate_data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = DATA_DIR / "pharmamate.sqlite3"
print("영구 저장 폴더:", DATA_DIR)
print("SQLite DB 위치:", DB_PATH)

## 4. 기존 핵심 엔진: LLM · 기사 추출 · 구조화 분석

원본의 **기사 URL/본문 추출, Pydantic Structured Output, ChatPromptTemplate** 구조를 유지했습니다. 기사를 빠르게 읽는 데 초점을 맞춰 핵심 용어는 최대 5개만 보여줍니다.

- URL 자동 추출이 막히거나 메뉴/광고가 섞이면 **기사 본문 직접 붙여넣기**를 사용하세요.
- 기술적 용어는 정의뿐 아니라 **이 기사에서의 맥락**을 함께 추출합니다.
- API 호출 전송량을 제한하기 위해 기사 입력이 너무 길면 일부만 분석합니다 (원문 읽기 화면에는 사용자가 입력한 전체 내용 유지).

In [ ]:
import re, html, json, hashlib, sqlite3, random
from typing import List, Literal, TypedDict
from urllib.parse import urlparse, parse_qsl, urlencode, urlunparse
import unicodedata
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import requests
from bs4 import BeautifulSoup

model = ChatOpenAI(model=MODEL_NAME, temperature=0.2, timeout=90, max_retries=2)

CATEGORIES = ["Modality", "Drug", "Target", "Biomarker", "Disease", "Clinical Trial",
              "Clinical Metric", "Regulatory", "Business", "Manufacturing", "Platform",
              "Diagnostic", "Mechanism", "Technology"]

class PharmaTerm(BaseModel):
    term: str = Field(description="핵심 전문용어. 약어가 등장하면 '정식 명칭 (AOC)'처럼 표기 가능")
    category: Literal["Modality", "Drug", "Target", "Biomarker", "Disease", "Clinical Trial",
                      "Clinical Metric", "Regulatory", "Business", "Manufacturing", "Platform",
                      "Diagnostic", "Mechanism", "Technology"]
    reason: str = Field(description="이 용어를 배워야 하는 이유 한 문장")
    simple_definition: str = Field(description="비전공자도 바로 이해할 수 있는 쉬운 한 문장 정의")
    article_context: str = Field(description="이 기사에서 이 용어가 사용된 맥락 한 문장")
    aliases: List[str] = Field(description="기사에 나타난 약어/전체 이름/한국어 동의어 (알려진 경우만)")

class ArticleAnalysis(BaseModel):
    article_type: Literal["임상", "규제", "기술/플랫폼", "투자", "기술이전", "파트너십",
                          "사업/전략", "연구/학술", "산업동향", "기타"]
    one_line_summary: str = Field(description="기사를 쉽게 이해할 수 있게 핵심만 1~2문장")
    key_issue: str = Field(description="기업·기술·사건과 연결되는 업계 핵심 이슈 1~2문장")
    key_entities: List[str] = Field(description="기사 속 주요 기업, 신약 후보, 연구기관 등")
    key_terms: List[PharmaTerm] = Field(description="앞으로도 자주 접할 핵심 개념 3~5개")
    confusing_points: List[str] = Field(description="읽을 때 혼동하기 쉬운 지점 최대 2개")

article_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 비전공자를 위한 제약·바이오 뉴스 읽기 튜터입니다.
목표는 기사를 대신 읽어주는 것이 아니라 사용자가 기사 원문을 빠르게 읽게 돕는 것입니다.
규칙:
1. 기사를 장황하게 요약하지 말고 업계에서 실제로 일어난 일을 쉽고 정확하게 정리합니다.
2. 회사명·제품명 등 고유명사와 재사용 가치가 있는 일반 전문용어를 구분하세요.
3. 재사용 가치가 높은 핵심 용어 3~5개만 골라 simple_definition과 article_context를 짧게 작성하세요.
4. 기사에 약어/정식 명칭이 함께 제시되면 aliases에 함께 기록합니다. 뜻을 모르면 추측하지 않습니다.
5. 본문에 없는 사실, 확인되지 않은 승인·임상 결과를 사실처럼 만들지 마세요.
6. 회사 주장, 전망, 검증된 결과를 구분합니다.
7. 기사의 원문 중 웹사이트 탐색 메뉴·광고·관련 기사 목록 등은 핵심 근거로 쓰지 않습니다.
8. category는 기존 PharmaMate의 Category 중 가장 적절한 하나를 선택합니다."""),
    ("human", "[기사 본문]\n{article}")
])
article_chain = article_prompt | model.with_structured_output(ArticleAnalysis)

def valid_remote_url(url: str) -> bool:
    parsed = urlparse(url)
    if parsed.scheme not in ("https", "http") or not parsed.netloc:
        return False
    # 개인 실행용 앱이지만 로컬/사설 네트워크 URL 접근을 방지합니다.
    host = (parsed.hostname or "").lower()
    if host in ("localhost",) or host.endswith(".local"):
        return False
    import ipaddress
    try:
        ip = ipaddress.ip_address(host)
        if not ip.is_global:
            return False
    except ValueError:
        pass
    return True

ARTICLE_BODY_SELECTORS = [
    '#article-view-content-div', '[itemprop="articleBody"]',
    '#articleBody', '#article_body', '#article-body',
    '.article-view-content', '.article_view_content', '.article-body',
    '.article_body', '.news-view-content', '.article-content', '.article_view', 'article'
]
ARTICLE_NOISE_SELECTORS = [
    'script', 'style', 'noscript', 'iframe', 'svg', 'form', 'button',
    'input', 'textarea', 'select', 'nav', 'footer', 'aside',
    '[id*="comment"]', '[class*="comment"]',
    '[id*="reply"]', '[class*="reply"]',
    '.article-tools', '.article-toolbar', '.related-news', '.share', '.sns'
]

def _clean_article_node(node) -> str:
    fragment=BeautifulSoup(str(node),'html.parser')
    for selector in ARTICLE_NOISE_SELECTORS:
        for unwanted in fragment.select(selector):
            unwanted.decompose()
    lines=[]
    for line in fragment.get_text('\n',strip=True).splitlines():
        line=re.sub(r'\s+',' ',line).strip()
        if line and (not lines or line != lines[-1]):
            lines.append(line)
    return '\n'.join(lines)

def _jsonld_article_body(value):
    if isinstance(value,dict):
        body=value.get('articleBody') or value.get('articlebody')
        if isinstance(body,str) and len(body.strip())>=300:
            return body.strip()
        for nested in value.values():
            found=_jsonld_article_body(nested)
            if found:
                return found
    elif isinstance(value,list):
        for nested in value:
            found=_jsonld_article_body(nested)
            if found:
                return found
    return ''

def extract_article_text(soup):
    # 기사 본문을 감싸는 요소만 살펴보고 댓글·버튼·페이지 메뉴는 제외합니다.
    seen=set()
    for selector in ARTICLE_BODY_SELECTORS:
        for node in soup.select(selector):
            text=_clean_article_node(node)
            if len(text)>=300 and text not in seen:
                return text,'article_selector'
            seen.add(text)
    # 일부 사이트는 본문을 JSON-LD 메타데이터에만 넣습니다.
    for script in soup.select('script[type="application/ld+json"]'):
        try:
            body=_jsonld_article_body(json.loads(script.string or script.get_text()))
            if body:
                return _clean_article_node(BeautifulSoup(body,'html.parser')),'json_ld'
        except (TypeError,ValueError,json.JSONDecodeError):
            continue
    raise ValueError('페이지에서 기사 본문 영역을 찾지 못했습니다. 기사 본문을 직접 붙여넣어 주세요.')

def _page_title(soup):
    for selector in ['meta[property="og:title"]','meta[name="twitter:title"]']:
        node=soup.select_one(selector)
        if node and node.get('content'):
            return node['content'].strip()
    node=soup.find('h1') or soup.find('title')
    return node.get_text(' ',strip=True) if node else ''

def load_article(url: str = "", pasted_text: str = ""):
    url, pasted_text = (url or "").strip(), (pasted_text or "").strip()
    if pasted_text:
        if len(pasted_text) < 120:
            raise ValueError("기사 본문이 너무 짧습니다. 본문을 충분히 붙여넣어 주세요.")
        return pasted_text, {"source": url or "직접 입력", "title": "(제목 없음)", "load_method": "manual"}
    if not url:
        raise ValueError("기사 URL 또는 기사 본문이 필요합니다.")
    if not valid_remote_url(url):
        raise ValueError("올바른 공개 http(s) 기사 URL을 입력해 주세요.")
    try:
        response=requests.get(url,headers={
            'User-Agent':'Mozilla/5.0 (compatible; PharmaMate/2.2)',
            'Accept-Language':'ko-KR,ko;q=0.9,en;q=0.8'
        },timeout=(10,30))
        response.raise_for_status()
        if not valid_remote_url(response.url):
            raise ValueError('기사 주소가 허용되지 않는 위치로 연결되었습니다.')
        soup=BeautifulSoup(response.content,'html.parser')
        text,method=extract_article_text(soup)
        metadata={'source':url,'title':_page_title(soup),'load_method':method}
        return text,metadata
    except ValueError:
        raise
    except Exception as exc:
        raise ValueError("URL에서 기사 본문을 가져오지 못했습니다. 기사 본문을 직접 붙여넣어 주세요. "
                         f"(원인: {type(exc).__name__})") from exc

def analyze_article(article_text: str) -> ArticleAnalysis:
    return article_chain.invoke({"article": article_text[:24000]})

def article_id_for(text: str, url: str = "") -> str:
    return hashlib.sha256(((url or "") + "\n" + text).encode("utf-8")).hexdigest()[:20]

print("기사 분석 모델 및 구조화 출력 준비 완료")

## 5. 기존 용어 학습 기능 재사용 + 쉬운 설명

기사 미리보기에는 **한 줄 정의**만 표시하고, 선택한 용어를 더 알고 싶을 때만 LLM을 한 번 더 호출해 4단계 설명을 제공합니다.

In [ ]:
term_intro_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 제약·바이오 산업 뉴스를 처음 읽는 비전공자에게 설명하는 튜터입니다.
딱 다음 순서로 간결하게 설명하세요.
1. 한 줄 정의
2. 쉽게 말하면 (생활 속 비유 또는 쉬운 예시)
3. 이 기사에서는 (기사에서 확인되는 사실만)
4. 지금 기억할 한 가지
확실하지 않은 사실은 모른다고 말하고, 원문의 주장과 검증된 사실을 구분합니다."""),
    ("human", "[기사]\n{article}\n\n[선택한 용어]\n{term}")
])
term_intro_chain = term_intro_prompt | model | StrOutputParser()

def explain_term(article_text: str, term: str) -> str:
    return term_intro_chain.invoke({"article": article_text[:14000], "term": term})

## 6. SQLite 학습 Memory: 용어·읽은 기사·산업 이슈

- `learned_terms`: 내가 **학습 기록 저장**을 누른 용어
- `term_examples`: 용어가 어떤 기사에서 등장했는지 기록 (기사별 중복 방지)
- `read_articles`: 내가 **읽기 완료**를 누른 기사 및 당시의 산업 이슈

동일한 AOC 약어가 여러 방식으로 표기되어도 되도록 `normalize_term()`에서 **괄호 속 약어를 우선** 처리합니다. 같은 약어가 다른 뜻을 가진 경우에는 수동 검토가 필요합니다.

In [ ]:
def normalize_term(term: str) -> str:
    text = (term or "").strip()
    acronyms = re.findall(r"\(([A-Za-z][A-Za-z0-9+-]{1,15})\)", text)
    if acronyms:
        return acronyms[-1].casefold()
    if re.fullmatch(r"[A-Za-z][A-Za-z0-9+-]{1,15}", text):
        return text.casefold()
    return re.sub(r"[\s_\-]+", "", text).casefold()

def aliases_for(term: dict) -> List[str]:
    names = [term.get("term", "")] + list(term.get("aliases") or [])
    names.extend(re.findall(r"\(([A-Za-z][A-Za-z0-9+-]{1,15})\)", term.get("term", "")))
    if "(" in term.get("term", ""):
        names.append(term["term"].split("(")[0].strip())
    return list(dict.fromkeys(x.strip() for x in names if x and x.strip()))

def connect_db():
    conn = sqlite3.connect(str(DB_PATH), timeout=15)
    conn.row_factory = sqlite3.Row
    return conn

def initialize_database():
    with connect_db() as conn:
        conn.executescript("""
        CREATE TABLE IF NOT EXISTS learned_terms (
            term_key TEXT PRIMARY KEY,
            term_name TEXT NOT NULL,
            aliases_json TEXT NOT NULL,
            category TEXT NOT NULL,
            simple_definition TEXT NOT NULL,
            first_learned_at TEXT NOT NULL,
            last_reviewed_at TEXT,
            review_count INTEGER NOT NULL DEFAULT 0
        );
        CREATE TABLE IF NOT EXISTS read_articles (
            article_id TEXT PRIMARY KEY,
            title TEXT NOT NULL,
            source_url TEXT,
            article_type TEXT,
            summary TEXT,
            key_issue TEXT,
            entities_json TEXT NOT NULL,
            read_at TEXT NOT NULL
        );
        CREATE TABLE IF NOT EXISTS term_examples (
            term_key TEXT NOT NULL,
            article_id TEXT NOT NULL,
            context TEXT,
            source_url TEXT,
            created_at TEXT NOT NULL,
            PRIMARY KEY(term_key, article_id)
        );
        """)

def now_local():
    return datetime.now().astimezone().isoformat(timespec="seconds")

def get_learned_rows():
    with connect_db() as conn:
        return [dict(x) for x in conn.execute("SELECT * FROM learned_terms ORDER BY first_learned_at DESC")]

def get_saved_term(term: dict):
    keys = {normalize_term(x) for x in aliases_for(term)}
    for row in get_learned_rows():
        other = {row['term_key']} | {normalize_term(x) for x in json.loads(row['aliases_json'])}
        if keys & other:
            return row
    return None

def normalize_article_title(title: str) -> str:
    text=unicodedata.normalize('NFKC',str(title or '')).casefold().strip()
    parts=re.split(r'\s*<\s*',text,maxsplit=1)
    if len(parts)>1 and any(tag in parts[1] for tag in ('기사본문','제약','바이오','산업','기업')):
        text=parts[0]
    text=re.sub(r'\s*[-–—|｜]\s*(?:팜뉴스|히트뉴스|메디파나뉴스|약업신문|의학신문|데일리팜|뉴스1|뉴시스|연합뉴스)\s*$','',text)
    return re.sub(r'[^0-9a-z가-힣]+','',text)

def article_titles_match(left: str, right: str) -> bool:
    if not left or not right or min(len(left),len(right))<10:
        return False
    if left==right:
        return True
    shorter,longer=sorted((left,right),key=len)
    return len(shorter)>=24 and longer.startswith(shorter) and len(shorter)/len(longer)>=0.75

def canonical_article_url(url: str) -> str:
    try:
        parsed=urlparse(str(url or '').strip())
        if parsed.scheme.lower() not in ('http','https') or not parsed.hostname:
            return ''
        host=parsed.hostname.lower()
        if host.startswith('www.'):
            host=host[4:]
        port=parsed.port
        if port and not ((parsed.scheme.lower()=='http' and port==80) or
                         (parsed.scheme.lower()=='https' and port==443)):
            host=f'{host}:{port}'
        tracking={'fbclid','gclid','dclid','mc_cid','mc_eid','_ga'}
        query=[(key,value) for key,value in parse_qsl(parsed.query,keep_blank_values=True)
               if not key.lower().startswith('utm_') and key.lower() not in tracking]
        query.sort()
        return urlunparse((parsed.scheme.lower(),host,parsed.path.rstrip('/') or '/',
                           '',urlencode(query,doseq=True),''))
    except ValueError:
        return ''

def save_term_card(term: dict, article_id: str, article_url: str):
    prev = get_saved_term(term)
    key = prev["term_key"] if prev else normalize_term(term["term"])
    if not key:
        raise ValueError("저장할 용어가 없습니다.")
    aliases = list(dict.fromkeys((json.loads(prev['aliases_json']) if prev else []) + aliases_for(term)))
    with connect_db() as conn:
        conn.execute("""INSERT INTO learned_terms
           (term_key,term_name,aliases_json,category,simple_definition,first_learned_at)
           VALUES (?,?,?,?,?,?)
           ON CONFLICT(term_key) DO UPDATE SET aliases_json=excluded.aliases_json,
           category=excluded.category, simple_definition=excluded.simple_definition""",
           (key, prev['term_name'] if prev else term['term'], json.dumps(aliases,ensure_ascii=False),
            term['category'], term['simple_definition'], now_local()))
        conn.execute("""INSERT OR IGNORE INTO term_examples
           (term_key,article_id,context,source_url,created_at) VALUES (?,?,?,?,?)""",
           (key, article_id, term.get('article_context',''), article_url, now_local()))
    return bool(prev), key

def save_read_article(state: dict):
    analysis = state.get('analysis') or {}
    aid = state.get('article_id')
    if not aid or not analysis:
        raise ValueError("먼저 기사를 분석해 주세요.")
    title=state.get('title','기사')
    source_url=state.get('url','')
    title_key=normalize_article_title(title)
    source_key=canonical_article_url(source_url)
    with connect_db() as conn:
        existing=conn.execute('SELECT title,source_url FROM read_articles').fetchall()
        for saved in existing:
            same_url=bool(source_key and source_key==canonical_article_url(saved['source_url']))
            same_title=article_titles_match(title_key,normalize_article_title(saved['title']))
            if same_url or same_title:
                return False
        cur = conn.execute("""INSERT OR IGNORE INTO read_articles
          (article_id,title,source_url,article_type,summary,key_issue,entities_json,read_at)
          VALUES (?,?,?,?,?,?,?,?)""",
          (aid,title,source_url,analysis.get('article_type',''),
           analysis.get('one_line_summary',''),analysis.get('key_issue',''),
           json.dumps(analysis.get('key_entities',[]),ensure_ascii=False),now_local()))
        if cur.rowcount != 1:
            return False
        for term in analysis.get('key_terms', []):
            prev = get_saved_term(term)
            if prev:
                conn.execute("""INSERT OR IGNORE INTO term_examples
                  (term_key,article_id,context,source_url,created_at) VALUES (?,?,?,?,?)""",
                  (prev['term_key'],aid,term.get('article_context',''),state.get('url',''),now_local()))
    return cur.rowcount == 1

def get_recent_articles(limit=20):
    with connect_db() as conn:
        return [dict(x) for x in conn.execute(
            "SELECT * FROM read_articles ORDER BY read_at DESC LIMIT ?", (limit,))]

def get_term_examples(term_key, limit=8):
    with connect_db() as conn:
        return [dict(x) for x in conn.execute(
            "SELECT * FROM term_examples WHERE term_key=? ORDER BY created_at DESC LIMIT ?", (term_key,limit))]

def search_study_memory(question: str, limit=5):
    question_low = (question or '').casefold()
    question_tokens = [s for s in re.findall(r'[A-Za-z0-9가-힣]+',question_low) if len(s)>1]
    ranked = []
    for r in get_learned_rows():
        names = json.loads(r['aliases_json']) + [r['term_name']]
        score = sum(4 for n in names if len(n.strip())>=2 and n.casefold() in question_low)
        score += sum(1 for t in question_tokens if t in r['simple_definition'].casefold())
        if score:
            ranked.append((score, r))
    ranked.sort(key=lambda x:x[0], reverse=True)
    return [r for _,r in ranked[:limit]]

initialize_database()
print("SQLite DB 초기화 완료:", DB_PATH)

# PharmaMate 2.2: 기존 DB를 보존하면서 학습 메모와 관련 질문·답변을 추가 저장합니다.
def initialize_study_notes():
    with connect_db() as conn:
        conn.executescript("""
        CREATE TABLE IF NOT EXISTS term_notes (
            term_key TEXT PRIMARY KEY,
            note TEXT NOT NULL,
            updated_at TEXT NOT NULL
        );
        CREATE TABLE IF NOT EXISTS term_qa (
            qa_id TEXT PRIMARY KEY,
            term_key TEXT NOT NULL,
            article_id TEXT NOT NULL DEFAULT '',
            question TEXT NOT NULL,
            answer TEXT NOT NULL,
            created_at TEXT NOT NULL
        );
        CREATE INDEX IF NOT EXISTS idx_term_qa_key ON term_qa(term_key,created_at);
        """)

def get_saved_note(term_key: str) -> str:
    with connect_db() as conn:
        row=conn.execute('SELECT note FROM term_notes WHERE term_key=?',(term_key,)).fetchone()
    return row['note'] if row else ''

def get_term_qa(term_key: str, limit: int = 15):
    with connect_db() as conn:
        return [dict(r) for r in conn.execute(
            'SELECT * FROM term_qa WHERE term_key=? ORDER BY created_at DESC LIMIT ?',
            (term_key,limit))]

def related_qa_in_state(state: dict, term: dict):
    # 저장 시점에 선택된 용어와 실제로 연결되었다고 판정된 대화만 사용합니다.
    prev=get_saved_term(term)
    keys={normalize_term(x) for x in aliases_for(term)}
    if prev:
        keys.add(prev['term_key'])
    return [item for item in (state or {}).get('history',[]) if
            item.get('linked_term_key') in keys and item.get('question') and item.get('answer')]

def draft_learning_note(term: dict, state: dict) -> str:
    relevant=related_qa_in_state(state,term)
    lines=[f"## {term['term']}",
           f"- 분류: {term.get('category','기타')}",
           f"- 기본 개념: {term.get('simple_definition','')}",
           f"- 기사에서의 의미: {term.get('article_context','')}"]
    if relevant:
        lines += ['', '### 관련 질문·답변']
        for qa in relevant[-8:]:
            lines += [f"**Q.** {qa['question']}",f"**A.** {qa['answer'][:1600]}", '']
    return '\n'.join(lines)

def save_learning_bundle(term: dict, state: dict, user_note: str):
    # 기존 learned_terms/term_examples 로직을 재사용합니다.
    prev,key=save_term_card(term,state.get('article_id','') or '',state.get('url',''))
    relevant=related_qa_in_state(state,term)
    note=(user_note or '').strip() or draft_learning_note(term,state)
    with connect_db() as conn:
        conn.execute("""INSERT INTO term_notes(term_key,note,updated_at) VALUES (?,?,?)
            ON CONFLICT(term_key) DO UPDATE SET note=excluded.note,updated_at=excluded.updated_at""",
            (key,note,now_local()))
        inserted=0
        for item in relevant:
            # 같은 대화가 여러 번 저장되어도 중복 삽입하지 않습니다.
            data=(key+(item.get('article_id') or '')+item['question']+item['answer']).encode('utf-8')
            qa_id=hashlib.sha256(data).hexdigest()[:32]
            cur=conn.execute("""INSERT OR IGNORE INTO term_qa
                (qa_id,term_key,article_id,question,answer,created_at) VALUES(?,?,?,?,?,?)""",
                (qa_id,key,item.get('article_id') or '',item['question'],item['answer'],now_local()))
            inserted+=cur.rowcount
    return prev,key,inserted,len(relevant),note

initialize_study_notes()
print('학습 메모와 Q&A 저장 테이블 준비 완료')


## 7. 기사 읽기 뷰: 용어 강조 + 학습 여부 표시

기존 Gradio의 기사 브리핑 옆에 **본문 전체**를 보여줍니다. 핵심 용어는 본문에서 강조하고 마우스를 올리면 간단 설명을 확인할 수 있습니다. 기사 본문이 너무 길거나 잘못 추출되었으면 직접 입력으로 교체하는 것이 정확합니다.

In [ ]:
def _find_term_match(term_dicts: List[dict]):
    lookup = {}
    for term in term_dicts:
        for alias in aliases_for(term):
            if len(alias)<2 or len(alias)>70:
                continue
            lookup.setdefault(alias.casefold(), term)
    # 긴 표현부터 우선 매칭
    keys = sorted(lookup, key=len, reverse=True)
    if not keys:
        return None, lookup
    alternatives = []
    for k in keys:
        escaped = re.escape(k)
        if re.fullmatch(r'[a-z0-9+-]+',k):
            escaped = rf'(?<![A-Za-z0-9]){escaped}(?![A-Za-z0-9])'
        alternatives.append(escaped)
    return re.compile('|'.join(alternatives), flags=re.IGNORECASE), lookup

def render_article_html(article_text: str, term_dicts: List[dict]) -> str:
    pattern, lookup = _find_term_match(term_dicts)
    parts, pos = [], 0
    if pattern:
        for m in pattern.finditer(article_text):
            if m.start() < pos: continue
            parts.append(html.escape(article_text[pos:m.start()]))
            term = lookup.get(m.group().casefold())
            if term:
                known = bool(get_saved_term(term))
                bg = '#bbf7d0' if known else '#fde68a'
                title = html.escape(term.get('simple_definition',''), quote=True)
                parts.append(f'<mark title="{title}" style="background:{bg};border-radius:3px;padding:0 2px">'
                             f'{html.escape(m.group())}</mark>')
            else:
                parts.append(html.escape(m.group()))
            pos = m.end()
    parts.append(html.escape(article_text[pos:]))
    text = ''.join(parts)
    return ("""<div style="font-size:14px;color:inherit;line-height:1.95;white-space:pre-wrap;
            word-break:break-word;max-height:630px;overflow-y:auto;border:1px solid #d4d4d8;
            border-radius:12px;padding:18px;background:transparent">""" + text + '</div>')

def markdown_escape(text: str) -> str:
    return str(text or '').replace('|',r'\|').replace('\n',' ')

def build_preview(analysis: dict) -> str:
    lines = ['### 읽기 전에 알아둘 용어', '']
    for index, t in enumerate(analysis.get('key_terms',[])[:5],1):
        prev = get_saved_term(t)
        status = '**복습 용어**' if prev else '**새 용어**'
        lines.append(f"**{index}. {t['term']}** ({t['category']}) — {status}  ")
        lines.append(f"{t['simple_definition']}  ")
        lines.append(f"*이 기사에서는:* {t['article_context']}\n")
    return '\n'.join(lines)

def build_brief(analysis: dict, title=''):
    title = title or '기사'
    entities = ', '.join(analysis.get('key_entities',[])[:7]) or '확인된 주요 기업/약물 없음'
    return (f'### {title}\n\n'
            f"**기사 유형:** {analysis.get('article_type','')}\n\n"
            f"**한 줄 정리:** {analysis.get('one_line_summary','')}\n\n"
            f"**업계 이슈:** {analysis.get('key_issue','')}\n\n"
            f'**주요 기업·기술:** {entities}')

## 8. LangGraph Agent: 기사 / 학습 Memory / 최신 웹검색 라우팅

원본의 `SearchDecision` 구조를 확장해 **기사 질문, 저장된 학습 내용 질문, 최신 정보 질문**을 구분합니다. LLM이 경로를 정하고, LangGraph는 필요한 도구를 실행한 뒤 답변합니다.

- 기사 해석: 현재 본문으로 답변
- 이전에 공부한 용어: SQLite에서 학습 기록 검색
- 최근 뉴스·FDA 업데이트: Tavily 웹검색 (API 키 필요)
- 둘 다 필요: 학습 기록과 웹검색을 순서대로 호출

`web` 경로에 검색 API 키가 없으면 최신 사실을 검증했다고 표현하지 않습니다.

In [ ]:
import os
from tavily import TavilyClient
from backend.app.agents.graph import build_agent_graph

search_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"]) if os.getenv("TAVILY_API_KEY") else None
agent_graph = build_agent_graph(
    model=model,
    search_client=search_client,
    search_study_memory=search_study_memory,
    get_term_examples=get_term_examples,
    get_saved_note=get_saved_note,
)
print('Agent Graph 모듈 연결 완료: article / memory / web / memory_web')

## 9. Gradio UI 동작 함수 (2.2)

- `기사 읽기`는 원문과 브리핑에 집중합니다.
- `Agent 질문`에 선택 용어·설명·추천 질문·대화·학습 메모·저장을 모았습니다.
- Agent 라우터가 각 질문이 **그 당시 선택된 용어에 관련되는지**를 판단해 대화에 태그를 붙입니다. 다른 용어 질문은 언제든 가능합니다.
- **학습 메모 만들기**는 선택 용어에 연결된 대화만 요약하며, 사용자가 편집한 뒤 **이 용어 학습 기록 저장**을 눌러야 SQLite에 영구 저장됩니다.
- 이미 저장한 2.1 용어와 기사는 삭제하지 않습니다.


In [ ]:
def new_app_state():
    return {"url":"","article":"","title":"","analysis":None,
            "article_id":"","selected_term":"","history":[]}

def find_term(state, name):
    for t in (state.get('analysis') or {}).get('key_terms',[]):
        if t.get('term')==name:
            return t
    return None

def recommended_questions(term_name: str):
    if not term_name:
        return ["이 기사에서 중요한 개념을 쉬운 말로 설명해줘.",
                "제약·바이오 산업에서 자주 쓰이는 용어를 알려줘.",
                "최근 제약·바이오 분야에서 어떤 이슈를 봐야 해?"]
    return [f"{term_name}라는 개념을 일상생활 비유로 아주 쉽게 설명해줘.",
            f"지금 읽는 기사에서 {term_name}가 중요한 이유가 뭐야?",
            f"{term_name}와 혼동하기 쉬운 다른 개념은 무엇이고 차이는 뭐야?"]

def ui_pick_suggested_question(name, index):
    return recommended_questions(name)[index]

def ui_analyze_article(url,pasted_text,manual_title,state):
    state=new_app_state() # 다른 기사의 대화가 섞이지 않게 세션만 초기화, SQLite 유지
    try:
        article,meta=load_article(url,pasted_text)
        analysis=analyze_article(article).model_dump()
        analysis['key_terms']=analysis.get('key_terms',[])[:5]
        title=(manual_title or '').strip() or meta.get('title','') or '기사 원문'
        article_url=meta.get('source','')
        state.update({'article':article,'url':article_url,'title':title,'analysis':analysis,
                      'article_id':article_id_for(article,article_url),'history':[]})
        choices=[t['term'] for t in analysis.get('key_terms',[])[:5]]
        selected=choices[0] if choices else ''
        state['selected_term']=selected
        first=find_term(state,selected)
        prev=get_saved_term(first) if first else None
        saved_note=get_saved_note(prev['term_key']) if prev else ''
        note=('URL에서 기사 본문 영역만 추출했습니다. 원문과 다르면 본문을 직접 붙여넣어 주세요.'
              if meta.get('load_method') in ('article_selector','json_ld') else '직접 붙여넣은 본문을 사용했습니다.')
        return (build_brief(analysis,title),render_article_html(article,analysis['key_terms']),
                build_preview(analysis),gr.update(choices=choices,value=selected or None),
                '선택한 용어의 [쉽게 설명]을 눌러 주세요.',saved_note,state,
                note,[],f'**현재 기사:** {title}',title)
    except Exception as exc:
        return ('분석 실패','',f'오류: {exc}',gr.update(choices=[],value=None),
                '먼저 기사를 분석해 주세요.','',state,f'오류: {exc}',[],'기사 없음',manual_title or '')

def ui_update_article_title(title,state):
    state=state or new_app_state()
    if not state.get('analysis'):
        return gr.update(),gr.update(),state,gr.update()
    title=(title or '').strip() or '(제목 없음)'
    state['title']=title
    article_id=state.get('article_id','')
    if article_id:
        with connect_db() as conn:
            conn.execute('UPDATE read_articles SET title=? WHERE article_id=?',(title,article_id))
    return (build_brief(state['analysis'],title),f'**현재 기사:** {title}',state,
            f'기사 제목을 **{title}**(으)로 수정했습니다.')

def ui_select_term(name,state):
    state=state or new_app_state()
    state['selected_term']=name or ''
    t=find_term(state,name)
    prev=get_saved_term(t) if t else None
    saved_note=get_saved_note(prev['term_key']) if prev else ''
    return (state,'[선택 용어 쉽게 설명] 버튼을 누르세요.',saved_note,'')

def ui_explain_term(name,state):
    if not state or not state.get('article') or not name:
        return '먼저 기사를 분석하고 용어를 선택해 주세요.'
    try:
        return explain_term(state['article'],name)
    except Exception as exc:
        return f'설명 중 오류: {exc}'

def ui_generate_study_note(name,state):
    t=find_term(state or {},name)
    if not t:
        return '먼저 기사 분석 후 학습할 용어를 선택해 주세요.'
    default=draft_learning_note(t,state)
    qas=related_qa_in_state(state,t)
    if not qas:
        return default
    try:
        qa_text='\n\n'.join(f'Q: {q["question"]}\nA: {q["answer"][:1500]}' for q in qas[-8:])
        prompt=ChatPromptTemplate.from_messages([
            ('system', """당신은 개인 제약·바이오 학습 노트 편집자입니다.
아래는 사용자가 선택한 용어와 직접 관련된 질문·답변입니다.
반드시 그 내용과 용어 정의만 토대로 기억하기 좋은 Markdown 학습 메모를 작성하세요.
항목: 한 줄 개념 / 헷갈렸던 부분 Q&A / 기사에서 중요한 이유 / 기억할 점.
질문과 직접 관련 없는 내용은 넣지 마세요. 불확실한 의학·승인 정보는 단정하지 마세요.
간결하게 작성하고 원문 Q&A를 모두 복사하지 마세요."""),
            ('human','용어: {term}\n기본 정의: {definition}\n기사 맥락: {context}\n관련 대화:\n{qa}')
        ])
        return (prompt | model | StrOutputParser()).invoke({
            'term':t['term'],'definition':t['simple_definition'],
            'context':t['article_context'],'qa':qa_text})
    except Exception as exc:
        # 요약 API 오류가 있어도 기본 학습 메모를 편집·저장할 수 있습니다.
        return f'{default}\n\n> AI 요약을 만들지 못했습니다: {type(exc).__name__}'

def ui_save_term(name,note_text,state):
    t=find_term(state or {},name)
    if not t:
        return '먼저 용어를 선택해 주세요.','', '', note_text or ''
    try:
        was_known,key,new_count,matched_count,final_note=save_learning_bundle(t,state,note_text)
        msg=(f'저장 완료: **{t["term"]}** ({"기존 기록 업데이트" if was_known else "새 용어"})  '
             f'\n선택 용어와 관련된 현재 대화 **{matched_count}건** 중 신규 **{new_count}건** 저장. ')
        return (msg,build_preview(state['analysis']),
                render_article_html(state['article'],state['analysis']['key_terms']),final_note)
    except Exception as exc:
        return f'저장 실패: {type(exc).__name__}: {exc}',build_preview(state['analysis']),render_article_html(state['article'],state['analysis']['key_terms']),note_text

def ui_mark_read(state):
    if not state or not state.get('analysis'):
        return '먼저 기사를 분석해 주세요.'
    try:
        is_new=save_read_article(state)
        return '기사 읽기 완료. 산업 이슈가 기록되었습니다.' if is_new else '이미 읽은 기사입니다. 중복 저장하지 않았습니다.'
    except Exception as exc:
        return f'기록 실패: {exc}'

def ui_ask(question, history, state, use_article_context=True, record_option='자동 판단'):
    """기사·선택 용어와 관계없는 개념 질문도 답할 수 있습니다."""
    question = (question or '').strip()
    history = history or []
    state = state or new_app_state()
    if not question:
        return '', history+[{'role':'assistant','content':'질문을 입력해 주세요.'}], state
    try:
        article_context=(state.get('article') or '') if use_article_context else ''
        selected=(state.get('selected_term') or '') if use_article_context else ''
        record=find_term(state,selected) if selected else None
        term_context=selected
        if record:
            term_context+=(f"\n쉬운 설명: {record.get('simple_definition','')}"
                           f"\n현재 기사 속 맥락: {record.get('article_context','')}")
        recent=state.get('history',[])[-6:]
        hist_text='\n'.join(f"사용자: {item['question']}\nAI: {item['answer'][:800]}" for item in recent)
        result=agent_graph.invoke({'question':question,'article':article_context,
                                   'term':term_context,'history':hist_text})
        answer=result.get('answer') or '답변을 생성하지 못했습니다.'
        sources=result.get('sources') or []
        if sources:
            answer+='\n\n**참고 검색 출처**\n'+'\n'.join(
                f"- [{s['title'] or s['url']}]({s['url']})" for s in sources)
        linked=bool(record and use_article_context and result.get('related_to_selected',False))
        if record and use_article_context:
            if record_option=='선택 용어에 포함':
                linked=True
            elif record_option=='포함하지 않음':
                linked=False
        term_key=normalize_term(record['term']) if linked and record else ''
        state['history']=(state.get('history',[])+[{
            'question':question,'answer':answer,'linked_term_key':term_key,
            'article_id':state.get('article_id','') or ''
        }])[-20:]
        history=history+[{'role':'user','content':question},
                         {'role':'assistant','content':answer}]
        if result.get('route') in ('web','memory_web') and not sources:
            answer+='\n\n**주의:** 검색 결과가 없어 최신 사실은 별도 확인이 필요합니다.'
            history[-1]['content']=answer
            state['history'][-1]['answer']=answer
        return '',history,state
    except Exception as exc:
        error=f'질문 처리 오류: {type(exc).__name__}: {exc}'
        return question,history+[{'role':'assistant','content':error}],state

def ui_reset_chat(state):
    state=state or new_app_state()
    state['history']=[]
    return [],'',state

def ui_history_dashboard():
    rows=get_learned_rows()
    articles=get_recent_articles(30)
    if not rows:
        terms_text='아직 학습 기록이 없습니다. Agent 질문 탭에서 용어를 선택하고 [이 용어 학습 기록 저장]을 누르세요.'
    else:
        terms_text='| 학습 용어 | 분류 | 쉬운 설명 | 복습 횟수 |\n|---|---|---|---|\n'
        terms_text+='\n'.join(f"| {markdown_escape(r['term_name'])} | {r['category']} | "
               f"{markdown_escape(r['simple_definition'])} | {r['review_count']} |" for r in rows[:80])
    if not articles:
        article_text='아직 읽기 완료한 기사가 없습니다.'
    else:
        article_text='\n\n'.join(
          f"**{i+1}. {a['title']}** — {a['read_at'][:10]} / {a['article_type']}\n\n"
          f"핵심 이슈: {a['key_issue']}\n\n"
          f"주요 기업·기술: {', '.join(json.loads(a['entities_json']))}\n\n"
          f"출처: {a['source_url']}" for i,a in enumerate(articles))
    options=[r['term_name'] for r in rows]
    return terms_text,article_text,gr.update(choices=options,value=options[0] if options else None)

def ui_view_study_record(selected):
    row=next((r for r in get_learned_rows() if r['term_name']==selected),None)
    if not row:
        return '저장된 용어를 선택해 주세요.'
    note=get_saved_note(row['term_key'])
    qas=get_term_qa(row['term_key'],15)
    output=[f"### {row['term_name']}",
            f"**정의:** {row['simple_definition']}",
            '#### 내가 저장한 학습 메모',note or '별도로 작성한 학습 메모가 없습니다.',
            '#### 저장된 관련 질문·답변']
    if not qas:
        output.append('아직 이 용어와 연결해 저장한 질문·답변이 없습니다.')
    for i,qa in enumerate(qas,1):
        output.append(f"**{i}. Q:** {qa['question']}\n\n**A:** {qa['answer']}")
    return '\n\n'.join(output)

def ui_make_quiz():
    terms=get_learned_rows()
    if len(terms)<3:
        return '학습한 용어가 3개 이상이어야 퀴즈를 만들 수 있습니다.',gr.update(choices=[],value=None),{}
    correct=random.choice(terms)
    distractors=random.sample([r for r in terms if r['term_key']!=correct['term_key']],2)
    options=[correct['term_name']]+[r['term_name'] for r in distractors]
    random.shuffle(options)
    question=f"### 용어 맞히기\n**설명:** {correct['simple_definition']}\n\n이 설명에 해당하는 용어는?"
    return question,gr.update(choices=options,value=None),{'answer':correct['term_name'],
                                                             'term_key':correct['term_key']}

def ui_check_quiz(selected,quiz_state):
    if not quiz_state or not selected:
        return '퀴즈를 생성하고 답을 선택해 주세요.'
    is_correct=(selected==quiz_state['answer'])
    with connect_db() as conn:
        conn.execute("UPDATE learned_terms SET review_count=review_count+1,last_reviewed_at=? WHERE term_key=?",
                     (now_local(),quiz_state['term_key']))
    return ('정답입니다. ' if is_correct else '다시 복습해 보세요. ') + f"정답: **{quiz_state['answer']}**"

print('PharmaMate 2.2 UI 함수 준비 완료')


## 10. Gradio 2.2 — 3탭 화면 실행

화면 순서는 **(1) 기사 읽기 → (2) Agent 질문 → (3) 내 학습 기록**입니다.

- 기사 읽기 화면의 정보 과밀을 제거하고 **원문 + 핵심 용어** 중심으로 배치했습니다.
- Agent 질문 화면은 **왼쪽 용어 학습, 오른쪽 추천 질문·채팅**으로 나눴습니다.
- 다른 용어에 대한 질문도 자유롭게 가능하며, 저장 시 해당 용어와 연관된 대화만 학습 기록에 반영됩니다.
- 2.1의 Gradio가 계속 실행 중이라면 해당 셀 실행을 중지하거나 종료하세요. 화면이 비정상적으로 작다면 브라우저 확대/축소를 100%로 초기화하세요.


In [ ]:
import gradio as gr
import inspect

# 같은 커널에서 UI 셀을 다시 실행하면 이전 앱을 닫습니다.
if 'demo' in globals():
    try:
        demo.close()
    except Exception:
        pass

CUSTOM_CSS = """
.gradio-container { max-width: 1540px !important; width: min(98%, 1540px) !important;
    margin-left: auto !important; margin-right: auto !important; padding: 16px 20px !important; }
#reader, #study, #records {gap:14px !important;}
#article-input-panel {margin-bottom: 8px;}
#reading-area {min-height:540px;}
#chat-area {min-height:460px;}
#learning-panel, #qa-panel {min-width: 0;}
@media(max-width:850px) {.gradio-container {width:100% !important; padding:8px !important;}}
"""

blocks_kwargs={'title':'PharmaMate 2.2'}
# Gradio 6에서는 css 인자가 Blocks가 아니라 launch()로 이동했습니다.
if 'css' in inspect.signature(gr.Blocks.__init__).parameters:
    blocks_kwargs['css']=CUSTOM_CSS
if 'fill_width' in inspect.signature(gr.Blocks.__init__).parameters:
    blocks_kwargs['fill_width']=True

with gr.Blocks(**blocks_kwargs) as demo:
    gr.Markdown("""# PharmaMate 2.2
    **기사 원문을 읽고 → 질문하며 용어를 익히고 → 배운 내용을 저장하는 개인 학습 Agent**""")
    app_state=gr.State(new_app_state())
    quiz_state=gr.State({})

    with gr.Tab('1. 기사 읽기',elem_id='reader'):
        with gr.Group(elem_id='article-input-panel'):
            gr.Markdown('### 기사 입력')
            with gr.Row():
                url_input=gr.Textbox(label='기사 URL',placeholder='https://...',scale=3)
                title_input=gr.Textbox(label='기사 제목',scale=2,
                    info='분석 후에도 이 칸에서 제목을 바꾸면 화면과 읽기 기록에 반영됩니다.')
            with gr.Accordion('기사 본문 직접 붙여넣기 (선택)',open=False):
                article_input=gr.Textbox(label='본문 직접 입력 (입력하면 URL보다 우선)',lines=5)
            analyze_btn=gr.Button('기사 분석 → 읽기 시작',variant='primary')
            analyze_status=gr.Markdown('기사 URL 또는 본문을 입력하고 분석 버튼을 누르세요.')
        with gr.Row(equal_height=False):
            with gr.Column(scale=7,min_width=350):
                gr.Markdown('### 기사 원문')
                reading_html=gr.HTML('<div>기사를 분석하면 원문이 표시됩니다.</div>',elem_id='reading-area')
                read_btn=gr.Button('기사 읽기 완료 · 산업 이슈 저장',variant='secondary')
                read_status=gr.Markdown()
            with gr.Column(scale=5,min_width=320):
                gr.Markdown('### 기사 브리핑')
                briefing=gr.Markdown('기사를 분석하면 핵심 이슈와 주요 기업을 확인할 수 있습니다.')
                gr.Markdown('---')
                preview=gr.Markdown('### 읽기 전에 알아둘 용어\n\n기사를 분석하면 핵심 용어가 표시됩니다.')
                agent_tab_btn=gr.Button('Agent 질문 탭으로 이동',variant='primary')
                

    with gr.Tab('2. Agent 질문',elem_id='study'):
        current_article=gr.Markdown('기사 없이도 자유 질문이 가능합니다. 기사 분석 후에는 해당 기사를 참고합니다.')
        with gr.Row(equal_height=False):
            with gr.Column(scale=4,min_width=320,elem_id='learning-panel'):
                gr.Markdown('### 용어 학습')
                term_select=gr.Dropdown(label='학습할 용어 선택',choices=[],interactive=True)
                explain_btn=gr.Button('선택 용어 쉽게 설명',variant='secondary')
                term_intro=gr.Markdown('기사 분석 후 용어를 선택하면 쉬운 설명을 볼 수 있습니다.')
                gr.Markdown('---')
                gr.Markdown('### 내 학습 메모')
                gr.Markdown('선택 용어와 **관련된 질문·답변만** 요약해 메모를 작성할 수 있습니다. 저장 전 직접 수정할 수 있습니다.')
                generate_note_btn=gr.Button('대화 반영한 학습 메모 만들기')
                study_note=gr.Textbox(label='학습 메모 (수정 가능)',lines=8,placeholder='AI가 만든 메모를 수정하거나 직접 작성하세요. 비어 있어도 기본 설명과 관련 Q&A를 저장합니다.')
                save_term_btn=gr.Button('이 용어 학습 기록 저장',variant='primary')
                save_status=gr.Markdown()
            with gr.Column(scale=6,min_width=370,elem_id='qa-panel'):
                gr.Markdown('### AI 튜터와 자유 질문')
                gr.Markdown('선택한 용어뿐 아니라 **다른 용어, 기사 밖의 개념, 최신 동향**도 질문할 수 있습니다.')
                use_article_context=gr.Checkbox(label='현재 기사와 선택 용어를 답변에 참고',value=True)
                bot_config={'label':'PharmaMate AI 튜터 대화','height':420,'elem_id':'chat-area'}
                if 'type' in inspect.signature(gr.Chatbot.__init__).parameters:
                    bot_config['type']='messages'
                chatbot=gr.Chatbot(**bot_config)
                gr.Markdown('**선택한 용어에 대한 추천 질문** — 누르면 아래 입력창에 채워집니다.')
                with gr.Row():
                    q1=gr.Button('① 쉽게 비유해줘',size='sm')
                    q2=gr.Button('② 기사에서 왜 중요해?',size='sm')
                    q3=gr.Button('③ 비슷한 개념과 비교',size='sm')
                user_question=gr.Textbox(label='자유 질문',lines=2,
                    placeholder='예: CRL이 뭐야? / AOC랑 ADC의 차이는? / FDA의 최근 허가 기준은?')
                record_option=gr.Radio(
                    choices=['자동 판단','선택 용어에 포함','포함하지 않음'],
                    value='자동 판단',label='현재 질문을 선택 용어의 학습 기록에 포함')
                with gr.Row():
                    ask_btn=gr.Button('질문 보내기',variant='primary')
                    reset_chat_btn=gr.Button('대화 초기화')

    with gr.Tab('3. 내 학습 기록',elem_id='records'):
        refresh_btn=gr.Button('학습 기록 새로고침',variant='secondary')
        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown('### 저장한 용어')
                terms_dashboard=gr.Markdown('새로고침을 누르면 저장된 용어가 표시됩니다.')
            with gr.Column(scale=1):
                gr.Markdown('### 읽은 기사 · 산업 이슈')
                articles_dashboard=gr.Markdown('새로고침을 누르면 누적 이슈가 표시됩니다.')
        gr.Markdown('---')
        gr.Markdown('### 내 학습 메모와 질문·답변 다시 보기')
        saved_term_select=gr.Dropdown(label='저장된 용어 선택',choices=[])
        saved_study_detail=gr.Markdown('위 목록에서 저장된 용어를 선택하세요.')
        gr.Markdown('---')
        gr.Markdown('### 1문제 복습 퀴즈')
        quiz_btn=gr.Button('복습 문제 만들기')
        quiz_question=gr.Markdown()
        quiz_options=gr.Radio(label='정답 선택',choices=[])
        quiz_check=gr.Button('정답 확인')
        quiz_result=gr.Markdown()

    # 탭을 넘나들어도 동일한 app_state, 선택 용어, 대화를 공유합니다.
    agent_tab_btn.click(js="() => { document.getElementById('study-button')?.click(); }")
    analyze_btn.click(ui_analyze_article,
        inputs=[url_input,article_input,title_input,app_state],
        outputs=[briefing,reading_html,preview,term_select,term_intro,study_note,
                 app_state,analyze_status,chatbot,current_article,title_input])
    title_input.change(ui_update_article_title,inputs=[title_input,app_state],
                       outputs=[briefing,current_article,app_state,analyze_status])
    term_select.change(ui_select_term,inputs=[term_select,app_state],
                       outputs=[app_state,term_intro,study_note,save_status])
    explain_btn.click(ui_explain_term,inputs=[term_select,app_state],outputs=[term_intro])
    generate_note_btn.click(ui_generate_study_note,inputs=[term_select,app_state],outputs=[study_note])
    save_term_btn.click(ui_save_term,inputs=[term_select,study_note,app_state],
                        outputs=[save_status,preview,reading_html,study_note])
    read_btn.click(ui_mark_read,inputs=[app_state],outputs=[read_status])
    q1.click(lambda name:ui_pick_suggested_question(name,0),inputs=[term_select],outputs=[user_question])
    q2.click(lambda name:ui_pick_suggested_question(name,1),inputs=[term_select],outputs=[user_question])
    q3.click(lambda name:ui_pick_suggested_question(name,2),inputs=[term_select],outputs=[user_question])
    ask_inputs=[user_question,chatbot,app_state,use_article_context,record_option]
    ask_outputs=[user_question,chatbot,app_state]
    ask_btn.click(ui_ask,inputs=ask_inputs,outputs=ask_outputs)
    user_question.submit(ui_ask,inputs=ask_inputs,outputs=ask_outputs)
    reset_chat_btn.click(ui_reset_chat,inputs=[app_state],
                         outputs=[chatbot,user_question,app_state])
    refresh_btn.click(ui_history_dashboard,
                      outputs=[terms_dashboard,articles_dashboard,saved_term_select])
    saved_term_select.change(ui_view_study_record,inputs=[saved_term_select],outputs=[saved_study_detail])
    quiz_btn.click(ui_make_quiz,outputs=[quiz_question,quiz_options,quiz_state])
    quiz_check.click(ui_check_quiz,inputs=[quiz_options,quiz_state],outputs=[quiz_result])

# share=False: PC에서만 접속. 2.1 서버가 이미 7860을 쓰는 경우 7861 등으로 자동 변경될 수 있습니다.
launch_kwargs={'share':False,'server_name':'127.0.0.1','inbrowser':True,
               'inline':False,'show_error':True,'prevent_thread_lock':True}
if 'css' in inspect.signature(gr.Blocks.launch).parameters:
    launch_kwargs['css']=CUSTOM_CSS
demo.launch(**launch_kwargs)


## 11. 실행·저장 확인 순서

1. 프로젝트 폴더의 `.env`를 확인하고 **노트북 위에서 아래로 전부 실행**하세요.
2. **1. 기사 읽기:** URL/본문 입력 → 기사 분석 → 브리핑, 원문, 핵심 용어 확인.
3. **2. Agent 질문:** 드롭다운에서 학습할 용어 선택 → [선택 용어 쉽게 설명] 클릭.
4. 추천 질문 3개 중 하나를 눌러 입력창에 채운 후 전송하거나, 임의의 질문을 직접 입력하세요. **선택 용어와 무관한 질문도 가능합니다.**
5. 필요하면 고급 설정에서 이번 질문을 선택 용어 기록에 `포함`하거나 `포함하지 않음`으로 지정한 뒤 질문하세요.
6. 같은 용어에 대한 질문을 충분히 한 다음 [대화 반영한 학습 메모 만들기] → 메모 수정 → [이 용어 학습 기록 저장] 클릭. **저장 버튼을 누르기 전에는 대화 내용이 SQLite에 기록되지 않습니다.**
7. **3. 내 학습 기록:** [학습 기록 새로고침] → 저장한 용어 선택 → 학습 메모 및 실제 관련 Q&A 확인.
8. VS Code를 껐다가 다시 열어도 `pharmamate_data/pharmamate.sqlite3`에 기록이 남습니다.

### 확인 시나리오

- AOC 선택 → AOC에 관한 질문 후 저장 → `내 학습 기록`의 AOC에서 질문·답변 조회
- AOC 선택 → **다른 용어**(예: CRL) 질문 → Agent가 AOC와 무관한 질문으로 분류하면 AOC 기록에는 저장하지 않음
- AOC를 새 기사에서 재등장시켰을 때 복습 용어로 표시
- 동일한 관련 Q&A를 두 번 저장해도 `term_qa` 테이블에 중복 저장되지 않음
- 지난 버전의 학습 기록이 2.2에서도 그대로 표시되는지 확인

> **주의:** 관련성 판정은 LLM이 수행하므로 완벽하지 않을 수 있습니다. 꼭 기록해야 하는 질문은 고급 설정의 `선택 용어에 포함`, 기록하지 않을 질문은 `포함하지 않음`을 선택하세요. 저장 메모에 부정확한 내용을 넣지 않았는지도 확인하세요.


## 12. 회고 및 향후 개선점

### 실제 사용 목적에 맞춰 개선한 점

- 제약·바이오 기사의 전문용어 때문에 독해 시간이 길어지는 문제를 해결하기 위해 기사 원문과 핵심 용어 미리보기를 유지했습니다.
- **2.2에서 기사 읽기 / Agent 질문 / 내 학습 기록을 분리**해 지나치게 길던 단일 페이지 UI를 정리했습니다.
- 자유 질문 답변을 기사 선택 용어에 관련된 경우에만 연결하고, **학습 메모 + 실제 Q&A를 SQLite에 저장**하도록 확장했습니다.
- 관련성을 LLM이 자동 판단하되 사용자가 포함·제외를 수정할 수 있게 해 통제 가능성을 높였습니다.
- 실습 초기에는 Google Colab/Drive를 고려했으나, 장기적으로 개인이 사용할 목적이므로 VS Code 로컬 실행 + SQLite 영구 저장으로 전환했습니다.

### 향후 개선

1. **기사 자동 추천:** 관심 분야와 이미 학습한 용어를 기반으로 새로운 기사 추천, 읽은 기사 중복 제외
2. **기사·용어 클릭 연결:** 기사 속 강조된 용어를 클릭하면 바로 Agent 탭의 설명으로 이동
3. **복습 최적화:** 재등장 횟수, 오답, 시간 간격에 따른 복습 스케줄링
4. **대화 분류 개선:** 다중 용어 질문(예: ADC vs AOC)을 여러 용어에 함께 연결, 관련성 정확도 평가
5. **기사 신뢰성 강화:** 공식 규제 기관 자료와 업계 기사 간 사실 확인, 인용 자동 검증
6. **실행 방식 개선:** 노트북을 Python 모듈로 분리하고 VS Code에서 `python app.py`만으로 실행
7. **데이터 보호:** SQLite 자동 백업과 선택적 암호화. Google Drive 권한은 필수로 만들지 않음


## 13. VS Code 문제 해결

- **`ModuleNotFoundError`:** VS Code 오른쪽 위 **Select Kernel**이 `.venv` Python인지 확인하고 `python -m pip install -r requirements.txt`를 실행합니다. 설치 직후에는 커널을 재시작합니다.
- **`.env`가 없다는 오류:** `PROJECT_DIR` 출력 경로에 `.env`가 있는지 확인하세요. `.env.example` 파일의 이름을 바꾼 뒤 실제 API 키를 작성합니다.
- **`OPENAI_API_KEY`가 있는데도 오류:** `.env` 저장 후 설정 셀부터 재실행하세요. 오래된 환경변수값이 남아 있다면 커널을 재시작하세요.
- **Gradio 접속이 안 됨:** 마지막 셀 출력에 표시된 `http://127.0.0.1:<포트>`를 브라우저에서 직접 여세요. 포트 충돌 시 기존 서버를 종료하고 UI 셀을 재실행합니다.
- **기사 URL 추출 실패:** 해당 사이트의 차단/동적 렌더링 가능성이 있습니다. 본문을 복사해 입력해 주세요.
- **학습 기록이 사라진 것처럼 보임:** `DB_PATH` 출력 경로를 먼저 확인하세요. VS Code를 다른 작업 폴더로 열면 다른 DB가 생성될 수 있습니다. 기존 DB 파일을 삭제하지 마세요.
- **외부 검색이 안 됨:** `.env`에 `TAVILY_API_KEY`를 추가하고 API 설정 셀부터 다시 실행하세요.
- **서버 중지:** 마지막 셀의 실행 커널을 종료하거나 Gradio 앱 객체에서 `demo.close()`를 실행하세요.